# AudioCraft в Google Colab - Простая установка

Этот notebook поможет быстро запустить AudioCraft в Google Colab.

⚠️ **Важно:** Включите GPU (Runtime → Change runtime type → T4 GPU)

## 1. Проверка окружения

In [ ]:
import sys
print(f'Python: {sys.version}')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Установка ffmpeg

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg

# Устанавливаем dev-библиотеки для сборки пакета av
!apt-get install -y -qq libavformat-dev libavcodec-dev libavdevice-dev libavutil-dev libswscale-dev libavresample-dev

!ffmpeg -version | head -n 1
print('✓ FFmpeg и dev-библиотеки установлены')

## 3. Клонирование AudioCraft

In [ ]:
import os

# Клонируем если еще не клонировали
if not os.path.exists('/content/audiocraft'):
    !git clone https://github.com/facebookresearch/audiocraft.git
    print('✓ Репозиторий клонирован')
else:
    print('✓ Репозиторий уже существует')

# Переходим в папку
%cd /content/audiocraft

# Проверяем наличие setup.py
if os.path.exists('setup.py'):
    print('✓ setup.py найден')
    !ls -la setup.py
else:
    print('❌ setup.py не найден!')
    print('Содержимое текущей директории:')
    !pwd
    !ls -la

## 4. Установка зависимостей для Python 3.12

⚠️ **Примечание о предупреждениях numpy:**

После установки вы увидите предупреждения типа:
```
ERROR: ... requires numpy>=2, but you have numpy 1.26.4
```

**Это нормально и безопасно!** Конфликтующие пакеты (cupy, jax, opencv, tobler) не используются AudioCraft. Все необходимые компоненты будут работать корректно.

In [ ]:
# Обновляем pip
!pip install -q --upgrade pip setuptools wheel

print('Установка зависимостей для AudioCraft...')
print('⚠️  Вы увидите предупреждения о конфликтах numpy - это нормально!')
print('   Предустановленные пакеты Colab требуют numpy 2.x,')
print('   но AudioCraft работает только с numpy 1.x')
print('   Конфликтующие пакеты (opencv, jax, cupy) не используются AudioCraft.\n')

# Критичные пакеты для Python 3.12
!pip install -q 'numpy<2.0'
!pip install -q 'transformers>=4.35.0'
!pip install -q 'gradio>=4.0.0'
!pip install -q 'hydra-core>=1.3'

print('\n✓ Основные зависимости установлены')
print('✓ Предупреждения о numpy можно игнорировать')

In [ ]:
# Убедимся что мы в правильной директории
import os
print(f'Текущая директория: {os.getcwd()}')

if not os.path.exists('setup.py'):
    print('❌ Ошибка: setup.py не найден в текущей директории!')
    print('Попробуйте перезапустить с ячейки 3 (клонирование)')
else:
    # Установка пакета av (требует FFmpeg dev-библиотек)
    print('Установка av (Python bindings для FFmpeg)...')
    !pip install -q av
    
    # Установка AudioCraft
    print('Установка AudioCraft...')
    !pip install -q -e .
    print('✓ AudioCraft установлен')

## 5. Проверка установки

In [ ]:
print('Проверка критичных компонентов...')

import numpy as np
print(f'✓ NumPy: {np.__version__}')

import torch
print(f'✓ PyTorch: {torch.__version__}')
print(f'  CUDA: {torch.cuda.is_available()}')

import transformers
print(f'✓ Transformers: {transformers.__version__}')

import audiocraft
print(f'✓ AudioCraft: {audiocraft.__version__}')

from audiocraft.models import MusicGen
print('✓ MusicGen доступен')

from audiocraft.models import AudioGen  
print('✓ AudioGen доступен')

print('\n✅ Все компоненты работают корректно!')
print('   (Предупреждения о numpy можно игнорировать)')

## 6. Тестовая генерация музыки

In [ ]:
from audiocraft.models import MusicGen
from IPython.display import Audio

# Загружаем маленькую модель
print('Загрузка модели...')
model = MusicGen.get_pretrained('facebook/musicgen-small')
print(f'✓ Модель загружена: {model.name}')

In [ ]:
# Настройка параметров
model.set_generation_params(
    duration=8,
    temperature=1.0,
    top_k=250,
)

# Генерация
descriptions = ['upbeat electronic dance music with synthesizers']
print('Генерация...')
wav = model.generate(descriptions)

print(f'✓ Сгенерировано: {wav.shape}')

# Воспроизведение
display(Audio(wav[0].cpu().numpy(), rate=model.sample_rate))

## 7. Запуск Gradio интерфейса

Запустите веб-интерфейс для удобной генерации:

In [ ]:
!python -m demos.musicgen_app --share

## Готово! 🎉

Теперь можете:
- Генерировать музыку с разными описаниями
- Экспериментировать с параметрами
- Использовать другие модели (medium, large, melody)

### Полезные ссылки
- [Документация MusicGen](https://github.com/facebookresearch/audiocraft/blob/main/docs/MUSICGEN.md)
- [Документация AudioGen](https://github.com/facebookresearch/audiocraft/blob/main/docs/AUDIOGEN.md)